# Day 2 Instructor Demo. From a model to a measurement.

**Instructor notebook.** Runs on paid Colab Pro with GPU.

Five sections, matching the slide deck.

- **Section A.** Load the UN General Debate Corpus, take a stratified five-thousand-row sample.
- **Section B.** Truncate to 512 tokens, run zero-shot DeBERTa in batches.
- **Section C.** Save speech-level CSV, aggregate to country-year, write country-year CSV.
- **Section D.** Build the thirty-row gold set, compute confusion matrix, Cohen's kappa, Pearson r, Spearman rho, and a subgroup table.
- **Section E.** Plot climate-emphasis time series with a Paris Agreement marker. Region overlay.

**Off-the-shelf only.** No fine-tuning. The validation gap to a fine-tuned encoder is mentioned on slide 28 of the deck and not closed in this notebook.

**Pre-flight.** Run cells 1 through 4 once on the connected runtime so the corpus is downloaded and the model is cached. Disconnect and reconnect once to confirm warm starts.

## 0. Setup

In [ ]:
!pip install -q transformers==4.44.2 scikit-learn

In [ ]:
import os
import time
import urllib.request
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from transformers import pipeline, AutoTokenizer
from sklearn.metrics import cohen_kappa_score, accuracy_score, confusion_matrix
from scipy.stats import pearsonr, spearmanr

print('CUDA available', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device', torch.cuda.get_device_name(0))

---
## Section A. Load and sample

Download the full UN General Debate Corpus from the Harvard Dataverse. CC-zero. Stratify a five-thousand-row sample by year so coverage is balanced across decades.

In [ ]:
UNGDC_URL = (
    'https://dataverse.harvard.edu/api/access/datafile/'
    ':persistentId?persistentId=doi:10.7910/DVN/0TJX8Y/2CIIBR'
)
LOCAL_PATH = 'ungdc.csv'

if not os.path.exists(LOCAL_PATH):
    print('Downloading UNGDC...')
    urllib.request.urlretrieve(UNGDC_URL, LOCAL_PATH)

ungdc = pd.read_csv(LOCAL_PATH)
print('Shape', ungdc.shape)
print('Years', ungdc['year'].min(), 'to', ungdc['year'].max())
print('Countries', ungdc['country'].nunique())
print()
print(ungdc.head(2)[['country','year','session']])

In [ ]:
rng = np.random.default_rng(42)
TARGET_N = 5000

by_year = ungdc.groupby('year')
per_year = max(1, TARGET_N // by_year.ngroups)

sample_idx = []
for yr, grp in by_year:
    n = min(per_year, len(grp))
    sample_idx.extend(rng.choice(grp.index, size=n, replace=False).tolist())

sample = ungdc.loc[sample_idx].reset_index(drop=True)
print('Sampled', len(sample), 'speeches across', sample['year'].nunique(), 'years.')
print('Country count', sample['country'].nunique())

fig, ax = plt.subplots(figsize=(8, 2.5))
ax.bar(sample['year'].value_counts().sort_index().index,
       sample['year'].value_counts().sort_index().values,
       color='#5C7B6A', edgecolor='white')
ax.set_title('Per-year coverage in the stratified sample')
ax.set_xlabel('Year')
ax.set_ylabel('Speeches')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

---
## Section B. Truncate and score in batches

DeBERTa-v3-base has a 512-token context window. Speeches average two thousand words. We truncate every speech to its first 512 tokens for this demo. The choice belongs in your appendix.

In [ ]:
MODEL_ID = 'MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli'
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

def truncate_to_tokens(text, n=480):
    enc = tokenizer(text, truncation=True, max_length=n, return_tensors=None)
    return tokenizer.decode(enc['input_ids'], skip_special_tokens=True)

sample['text_short'] = sample['text'].astype(str).map(lambda t: truncate_to_tokens(t, 480))
print('Truncation done. Mean character length now:', int(sample['text_short'].str.len().mean()))

In [ ]:
zsc = pipeline(
    'zero-shot-classification',
    model=MODEL_ID,
    device=0 if torch.cuda.is_available() else -1,
)

candidate_labels = [
    'the speaker supports stronger international climate action',
    'the speaker is skeptical of international climate action',
    'the speaker does not discuss climate',
]

print('Pipeline ready. Three candidate labels.')

In [ ]:
t0 = time.time()
results = zsc(
    sample['text_short'].tolist(),
    candidate_labels=candidate_labels,
    batch_size=16,
)
elapsed = time.time() - t0
print(f'Scored {len(results)} speeches in {elapsed:.1f} sec.')

---
## Section C. Save and aggregate

Pull out the supportive-class probability and the argmax. Write speech-level CSV. Group to country-year. Write country-year CSV.

In [ ]:
SUPPORTIVE = candidate_labels[0]
SKEPTICAL = candidate_labels[1]
OFFTOPIC = candidate_labels[2]

def supportive_prob(r):
    return dict(zip(r['labels'], r['scores']))[SUPPORTIVE]

def top_short(r):
    return {SUPPORTIVE: 'supportive', SKEPTICAL: 'skeptical', OFFTOPIC: 'offtopic'}[r['labels'][0]]

scored = sample[['country', 'year', 'session']].copy()
scored['top_label'] = [top_short(r) for r in results]
scored['top_score'] = [r['scores'][0] for r in results]
scored['p_supportive'] = [supportive_prob(r) for r in results]

scored.to_csv('ungdc_speech_scored.csv', index=False)
print('Wrote ungdc_speech_scored.csv with', len(scored), 'rows.')
print()
print(scored.head())

In [ ]:
cy = (
    scored.assign(is_supportive=(scored['top_label'] == 'supportive').astype(int))
          .groupby(['country', 'year'], as_index=False)
          .agg(
              n_speeches=('top_label', 'size'),
              share_supportive=('is_supportive', 'mean'),
              mean_p_supportive=('p_supportive', 'mean'),
          )
)
cy.to_csv('ungdc_country_year.csv', index=False)
print('Country-year shape', cy.shape)
print(cy.head())

---
## Section D. The thirty-row gold set and the validation table

**Pedagogical note.** In a real research project, you would draw 30 random rows from `scored`, hand-code each, and use the hand-codes as the gold labels. To keep this demo runnable end-to-end without a coding session, we ship a small thirty-sentence gold set whose labels are bundled below. The validation logic is identical to what you would do on real hand-codes.

In [ ]:
gold_rows = [
    ('My country reaffirms its commitment to the Paris Agreement and calls on all states to raise their nationally determined contributions.', 'supportive'),
    ('Drought, rising seas, and the loss of arable land are existential threats to the small island nations represented in this hall.', 'supportive'),
    ('We reject any framework that imposes binding emissions targets on developing economies that did not cause the climate crisis.', 'skeptical'),
    ('The transition to a low-carbon economy must not come at the cost of energy poverty for our most vulnerable populations.', 'skeptical'),
    ('Today my delegation urges renewed support for the multilateral peacekeeping operations in the Sahel.', 'offtopic'),
    ('We remain deeply concerned by the humanitarian situation in our region and call for unimpeded access for relief workers.', 'offtopic'),
    ('My government welcomes the recent ratification of the Glasgow Climate Pact and pledges its full implementation.', 'supportive'),
    ('Investment in fossil-fuel infrastructure remains a national priority for energy security and industrial development.', 'skeptical'),
    ('The reform of the Security Council remains an urgent task that this Assembly must not defer for another decade.', 'offtopic'),
    ('We call upon all parties to recognize that climate finance promised by industrialized states has yet to be delivered in full.', 'supportive'),
    ('Our delegation calls for the immediate cessation of hostilities and a return to dialogue under UN auspices.', 'offtopic'),
    ('We will not accept a global climate regime that places the costs of adjustment on those least responsible for emissions.', 'skeptical'),
    ('We welcome the agreed loss-and-damage fund and stand ready to contribute our fair share.', 'supportive'),
    ('Net zero by 2050 must remain a shared multilateral commitment for every member state.', 'supportive'),
    ('My country has tripled its renewable-energy capacity since the last General Assembly.', 'supportive'),
    ('Trade barriers in agricultural markets continue to undermine food security in developing economies.', 'offtopic'),
    ('We mourn the lives lost to terrorism this year and reaffirm our commitment to the multilateral counter-terrorism framework.', 'offtopic'),
    ('We will continue to develop our hydrocarbon resources because that is the path our people have chosen.', 'skeptical'),
    ('Sustainable development requires a multilateral architecture that respects the principle of common but differentiated responsibilities.', 'supportive'),
    ('My country supports a comprehensive treaty on plastic pollution and strong language on phasing down fossil fuels.', 'supportive'),
    ('Health systems in our region remain under-resourced and the WHO must lead a renewed pandemic-preparedness effort.', 'offtopic'),
    ('Climate change is the existential challenge of our time, and inaction would be a betrayal of future generations.', 'supportive'),
    ('We do not accept that our development should be conditional on energy choices made by industrialized states.', 'skeptical'),
    ('Vienna Convention obligations on diplomatic immunity must be respected by all parties to this Assembly.', 'offtopic'),
    ('We urge all member states to support the upcoming review of the nationally determined contributions.', 'supportive'),
    ('My country reiterates its unwavering support for the two-state solution.', 'offtopic'),
    ('Adaptation finance must scale to match the increasing severity of climate impacts on vulnerable states.', 'supportive'),
    ('We welcome the Secretary-Generals proposal for a new financial pact for the global south.', 'offtopic'),
    ('We will resist any framework that uses climate ambition as a cover for new trade barriers.', 'skeptical'),
    ('Our coastal communities are already paying the price of decisions that were not theirs to make.', 'supportive'),
]

gold = pd.DataFrame(gold_rows, columns=['text', 'gold_label'])
print('Gold-set size', len(gold))
print(gold['gold_label'].value_counts())

In [ ]:
gold_results = zsc(gold['text'].tolist(), candidate_labels=candidate_labels, batch_size=16)
gold['pred_label'] = [
    {SUPPORTIVE: 'supportive', SKEPTICAL: 'skeptical', OFFTOPIC: 'offtopic'}[r['labels'][0]]
    for r in gold_results
]
gold['p_supportive'] = [supportive_prob(r) for r in gold_results]
gold['gold_score'] = gold['gold_label'].map({'supportive': 1, 'offtopic': 0, 'skeptical': -1})
gold['pred_score'] = gold['pred_label'].map({'supportive': 1, 'offtopic': 0, 'skeptical': -1})

print(gold[['gold_label', 'pred_label', 'p_supportive']].to_string())

In [ ]:
labels_order = ['supportive', 'skeptical', 'offtopic']
cm = confusion_matrix(gold['gold_label'], gold['pred_label'], labels=labels_order)

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cm, cmap='Greens')
ax.set_xticks(range(3))
ax.set_yticks(range(3))
ax.set_xticklabels(labels_order)
ax.set_yticklabels(labels_order)
ax.set_xlabel('predicted')
ax.set_ylabel('hand-coded gold')
ax.set_title('Confusion matrix on the 30-row gold set')
for i in range(3):
    for j in range(3):
        ax.text(j, i, str(cm[i, j]), ha='center', va='center', color='white' if cm[i,j] > cm.max()/2 else 'black')
plt.tight_layout()
plt.show()

In [ ]:
kappa = cohen_kappa_score(gold['gold_label'], gold['pred_label'])
acc = accuracy_score(gold['gold_label'], gold['pred_label'])
r, _ = pearsonr(gold['p_supportive'], gold['gold_score'])
rho, _ = spearmanr(gold['p_supportive'], gold['gold_score'])

print(f'Accuracy        {acc:.2f}')
print(f"Cohen's kappa   {kappa:.2f}")
print(f'Pearson r       {r:+.2f}     (supportive probability vs gold score)')
print(f'Spearman rho    {rho:+.2f}')

In [ ]:
# Subgroup table on the speech-level scored set, using simple keyword-based pseudo-gold for face validity.
# In a real project you would compute these on the actual hand-coded sample.
scored['era'] = np.where(scored['year'] < 2015, 'pre-2015', 'post-2015')
subgroup = (
    scored.groupby('era')
          .agg(n=('top_label', 'size'),
               share_supportive=('top_label', lambda s: (s == 'supportive').mean()),
               mean_top_score=('top_score', 'mean'))
          .round(3)
)
print(subgroup)

**What to point out.** Off-the-shelf zero-shot on this kind of three-way classification typically lands somewhere between 0.4 and 0.7 on Cohen's kappa. Whatever the live number is, treat it as the headline. Pearson r tends to run higher than kappa because the underlying probability is more sensitive than the argmax label.

---
## Section E. The time-series plot, with a Paris Agreement marker

In [ ]:
ts = (
    scored.assign(is_supportive=(scored['top_label'] == 'supportive').astype(int))
          .groupby('year', as_index=False)
          .agg(share_supportive=('is_supportive', 'mean'),
               mean_p_supportive=('p_supportive', 'mean'),
               n=('top_label', 'size'))
          .sort_values('year')
)

fig, ax = plt.subplots(figsize=(11, 4.5))
ax.plot(ts['year'], ts['mean_p_supportive'], color='#334B99', linewidth=2.4, marker='o', markersize=3, label='mean P(supportive)')
ax.axvline(2015, color='#2D493C', linestyle='--', alpha=0.7, linewidth=1.5)
ax.text(2015.2, ax.get_ylim()[1]*0.95, 'Paris Agreement', fontsize=10, color='#2D493C', fontweight='bold')
ax.set_xlabel('Year')
ax.set_ylabel('Mean supportive-class probability')
ax.set_title('Climate emphasis in UN General Assembly speeches')
ax.grid(True, alpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
# Region overlay using a coarse continent map. The mapping below is a teaching-grade approximation.
REGION_HINTS = {
    'EUR': ['FRA', 'DEU', 'GBR', 'ITA', 'ESP', 'NLD', 'POL', 'SWE', 'NOR', 'DNK', 'FIN', 'IRL', 'AUT', 'BEL', 'CHE'],
    'AFR': ['ZAF', 'NGA', 'KEN', 'EGY', 'GHA', 'DZA', 'MAR', 'TUN', 'SEN', 'CIV', 'ETH', 'TZA', 'UGA', 'ZMB'],
    'AMR': ['USA', 'CAN', 'MEX', 'BRA', 'ARG', 'CHL', 'COL', 'PER', 'VEN', 'CUB', 'JAM'],
    'ASI': ['CHN', 'JPN', 'KOR', 'IND', 'IDN', 'PAK', 'BGD', 'IRN', 'IRQ', 'TUR', 'SAU', 'THA', 'VNM', 'PHL'],
    'OCE': ['AUS', 'NZL', 'FJI', 'PNG', 'TUV', 'KIR', 'NRU', 'MHL', 'WSM', 'TON', 'VUT'],
}
code_to_region = {c: r for r, codes in REGION_HINTS.items() for c in codes}

scored_r = scored.copy()
scored_r['region'] = scored_r['country'].map(code_to_region).fillna('OTHER')
scored_r = scored_r[scored_r['region'] != 'OTHER']

regional = (
    scored_r.groupby(['region', 'year'], as_index=False)
            .agg(mean_p=('p_supportive', 'mean'))
)

fig, ax = plt.subplots(figsize=(11, 4.5))
palette = {'EUR': '#334B99', 'AFR': '#2D493C', 'AMR': '#5C7B6A', 'ASI': '#A8C9AE', 'OCE': '#B1F1B1'}
for region, sub in regional.groupby('region'):
    ax.plot(sub['year'], sub['mean_p'], marker='o', markersize=3, linewidth=2,
            label=region, color=palette.get(region, '#888'))
ax.axvline(2015, color='black', linestyle='--', alpha=0.4, linewidth=1.2)
ax.text(2015.2, ax.get_ylim()[1]*0.95, 'Paris', fontsize=9, alpha=0.7)
ax.set_xlabel('Year')
ax.set_ylabel('Mean P(supportive)')
ax.set_title('Regional split. Oceania often leads on climate language.')
ax.grid(True, alpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.legend(loc='upper left', frameon=False, ncol=5)
plt.tight_layout()
plt.show()

**What to point out.** The Oceania line is the visual hook. Small island states have been making climate central to their UN speeches since well before Paris. Europe joins them visibly post-2015. The pre-Paris versus post-Paris bend is a face-validity check, not a causal estimate.

---
## Wrap. What we just did

1. Loaded the UN General Debate Corpus and took a stratified five-thousand-row sample.
2. Truncated to 512 tokens and ran zero-shot DeBERTa with three candidate labels.
3. Wrote a speech-level CSV and a country-year CSV.
4. Computed Cohen's kappa, accuracy, Pearson r, and Spearman rho on a thirty-row gold set.
5. Plotted the climate-emphasis time series and a regional overlay.

**Day 3** introduces generative models. Same reproducibility demands. Different toolkit. Useful for tasks the encoder cannot easily express as a label.